In [1]:
!pip install open_spiel
!pip install pokerkit
!pip install networkx matplotlib

In [3]:
import pyspiel

# ---- Game Definition ----

class SamSeadGame(pyspiel.Game):

    def __init__(self):
        game_type = pyspiel.GameType(
            short_name="sam_sead",
            long_name="SAM SEAD Game",
            dynamics=pyspiel.GameType.Dynamics.SEQUENTIAL,
            chance_mode=pyspiel.GameType.ChanceMode.DETERMINISTIC,
            information=pyspiel.GameType.Information.IMPERFECT_INFORMATION,
            utility=pyspiel.GameType.Utility.ZERO_SUM,
            reward_model=pyspiel.GameType.RewardModel.TERMINAL,
            max_num_players=2,
            min_num_players=2,
            provides_information_state_string=True,
            provides_information_state_tensor=False,
            provides_observation_string=False,
            provides_observation_tensor=False
        )

        game_info = pyspiel.GameInfo(
            num_distinct_actions=2,
            max_chance_outcomes=0,
            num_players=2,
            min_utility=-2,
            max_utility=2,
            utility_sum=0,
            max_game_length=2
        )

        super().__init__(game_type, game_info, {})
    def new_initial_state(self):
        return SamSeadState(self)

    def make_py_observer(self, iig_obs_type=None, params=None):
        return None


# ---- Game State ----

class SamSeadState(pyspiel.State):

    def __init__(self, game):
        super().__init__(game)
        self.stage = 0
        self.radar = None
        self.history = []

    def current_player(self):

        if self.stage == 0:
            return 0  # SAM

        if self.stage == 1:
            return 1  # SEAD

        return pyspiel.PlayerId.TERMINAL

    def legal_actions(self, player=None):
        return [0,1]

    def apply_action(self, action):

        if self.stage == 0:
            self.radar = action
            self.stage = 1

        elif self.stage == 1:
            self.history.append(action)
            self.stage = 2

    def is_terminal(self):
        return self.stage == 2

    def returns(self):

        sead_action = self.history[0]

        if self.radar == 0 and sead_action == 0:
            return [2,-2]

        if self.radar == 0 and sead_action == 1:
            return [0,0]

        if self.radar == 1 and sead_action == 0:
            return [-2,2]

        if self.radar == 1 and sead_action == 1:
            return [0,0]

    def information_state_string(self, player):

        if player == 0:
            return f"Radar:{self.radar}"

        if player == 1:
            return "UnknownRadar"

        return ""

    def __str__(self):
        return f"Radar={self.radar}, History={self.history}"


# ---- Create Game ----

game = SamSeadGame()
state = game.new_initial_state()

print("Game created successfully")
print("Initial state:", state)

Game created successfully
Initial state: Radar=None, History=[]


In [4]:
state = game.new_initial_state()

print("Initial:", state)

# SAM chooses radar ON
state.apply_action(0)

print("After SAM action:", state)

# SEAD attacks
state.apply_action(0)

print("Terminal:", state)
print("Payoff:", state.returns())

Initial: Radar=None, History=[]
After SAM action: Radar=0, History=[]
Terminal: Radar=0, History=[0]
Payoff: [2, -2]


In [5]:
print(state.information_state_string(1))

UnknownRadar
